<a href="https://colab.research.google.com/github/KULDEEPSONI-source/DEEPLEARNING/blob/main/CNN_CODE_IMPLEMENT_PADDINGAND_STRIDES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D,Flatten
from keras.datasets import mnist


In [24]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

Same architecture as before, but one key change — `padding='same'` → `padding='valid'`, and no strides.

**What `padding='valid'` does:**
No padding is added. The kernel only slides over positions where it **fully fits**, so the output shrinks by `kernel_size - 1 = 2` pixels on each side per layer.

**Data flow:**

```
Input:   (28×28×1)
Conv1:   (28-2) × (28-2) = 26×26×32
Conv2:   (26-2) × (26-2) = 24×24×32
Conv3:   (24-2) × (24-2) = 22×22×32
Flatten: 22×22×32 = 15,488
Dense:   128
Output:  10
```

**vs the previous model:**

| | Previous (`same` + stride=2) | This (`valid`, stride=1) |
|---|---|---|
| Downsampling | Aggressive (stride) | Gentle (border trim only) |
| Final feature map | 4×4×32 = 512 | 22×22×32 = 15,488 |
| Parameters (Dense) | smaller | **much larger** |
| Spatial info retained | less | **more** |

> `valid` padding preserves more spatial detail but results in a much bigger Flatten output — so the Dense(128) layer here has **~30× more weights** to learn compared to the previous model.

In [25]:
model = Sequential()

model.add(Conv2D(32,kernel_size=(3,3),padding='valid', activation='relu', input_shape=(28,28,1)))
model.add(Conv2D(32,kernel_size=(3,3),padding='valid', activation='relu'))
model.add(Conv2D(32,kernel_size=(3,3),padding='valid', activation='relu'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(10,activation='softmax'))

In [26]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_22 (Conv2D)              │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 24, 24, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 22, 22, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 15488)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │     1,982,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,002,698 (7.64 MB)

 Trainable params: 2,002,698 (7.64 MB)

 Non-trainable params: 0 (0.00 B)

This builds a **CNN (Convolutional Neural Network)** for image classification — likely on MNIST (28×28 grayscale images, 10 classes).

**3 Conv2D Layers**
Each scans the image with 32 filters of size 3×3, using `stride=2` (jumps 2 pixels at a time, halving spatial size each layer) and `padding='same'` (keeps edges). ReLU drops negatives.

**Flatten**
Converts the 3D feature maps → 1D vector to feed into Dense layers.

**Dense(128) → Dense(10)**
Two fully connected layers. The final `softmax` outputs probabilities for each of the 10 digit classes.

**Data flow:**

```
(28×28×1) → Conv → (14×14×32) → Conv → (7×7×32) → Conv → (4×4×32) → Flatten → 512 → Dense(128) → Dense(10)
```

> The 3 strided convolutions act as a **learnable downsampler** — similar to MaxPooling but the model learns how to reduce size, rather than using a fixed rule.

In [27]:
model = Sequential()

model.add(Conv2D(32,kernel_size=(3,3),padding='same',strides=(2,2), activation='relu', input_shape=(28,28,1)))
model.add(Conv2D(32,kernel_size=(3,3),padding='same',strides=(2,2), activation='relu'))
model.add(Conv2D(32,kernel_size=(3,3),padding='same',strides=(2,2), activation='relu'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(10,activation='softmax'))

In [28]:
model.summary()


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_25 (Conv2D)              │ (None, 14, 14, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 7, 7, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 4, 4, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,770 (335.04 KB)

 Trainable params: 85,770 (335.04 KB)

 Non-trainable params: 0 (0.00 B)